# Measuring selection and read latency

We measure the time needed to select weight rows and read them into CPU memory. Paper greedy and Saturation tiles from the reference scale down to one row, plus a coarse branch share the activation trace, requested row budget and reader. The experiment records both elapsed time and retained activation importance, so a faster read can be assessed alongside what it retains.

All experiment-specific selection, tuning and measurement code is included below. `vlmflash` supplies the native Paper selector, cost-table interpolation and direct-I/O reader; their behavior is described where used. Inputs are `../01/block_estimates.csv` and `../02/activation_traces.npz`.

Run all cells from this directory on the target SSD with Linux, CUDA, PyTorch, a C++ compiler and the upstream dependencies installed. The save cells replace `measured_results.csv` and `measured_environment.json`. The companion [analysis notebook](03_02_analyze_selection_latency.ipynb) reads these outputs to produce tables and figures.

In [1]:
from pathlib import Path
from time import perf_counter_ns
from tempfile import TemporaryDirectory
import fcntl, json, os, platform, subprocess, sys
import numpy as np
import pandas as pd

PROFILE, TRACE = Path("../01/block_estimates.csv"), Path("../02/activation_traces.npz")
UPSTREAM = Path("../../vlm-flash")
RESULTS, ENVIRONMENT = Path("measured_results.csv"), Path("measured_environment.json")

The settings below define the sweep. The random seed fixes method order; timings are measured anew on each run.

In [2]:
MAX_TRACES_PER_SHAPE = 32  # 0 uses every archived trace.
FRACTIONS = np.arange(3, 11) / 10
WARMUP, REPEATS, THREADS = 3, 30, 6
PAPER, TILE = "Paper greedy (GPU sort)", "Tiles (1s)"
COARSE_SCALES = [1, 1.25, 1.5, 1.75, 2]

## Budget and retained importance

For a projection with $N$ rows, fraction $f$ gives $R=\max(1,\lfloor fN\rfloor)$. Archived importance is the mean absolute input activation, $v_i=\operatorname{mean}_{\rm batch,token}|x_i|$, normalized to mean one. A binary mask $M$ retains

$$q(M)=\frac{\sum_i v_iM_i}{\sum_i v_i},\qquad
\operatorname{fill}(M)=\frac{\sum_i M_i}{R}.$$

Paper may underfill; Saturation fills $R$. At $R=N$, both bypass selection. Zero-total importance leaves $q$ undefined. Up to `MAX_TRACES_PER_SHAPE` traces are sampled at evenly spaced archive positions within each shape.

In [3]:
with np.load(TRACE, allow_pickle=False) as archive:
    manifest = json.loads(str(archive["manifest_json"].item()))
    traces = [{**t, "values": archive[t["array"]]} for t in manifest["traces"]]
groups = [[t for t in traces if t["shape"] == shape] for shape in sorted({t["shape"] for t in traces})]
traces = [g[i] for g in groups for i in np.linspace(0, len(g)-1,
          min(len(g), MAX_TRACES_PER_SHAPE or len(g)), dtype=int)]
weight_bytes = {"float16": 2, "bfloat16": 2, "float32": 4}[manifest["model_dtype"]]

## Cost and tile granularity

The median profile defines $T(z)$, the cost in milliseconds of a $z$ KiB read. Costs interpolate between sizes, floor smaller reads at the minimum-size cost and extend the upper endpoint proportionally. A mask's cost sums $T$ over maximal selected runs. This serial estimate chooses masks; the measured reader uses multiple threads.

For an $N\times d$ weight matrix, a row occupies $w=d\,\text{bytes per weight}/1024$ KiB. Let $s_{\rm KiB}$ minimize measured cost per KiB, breaking ties toward smaller sizes. Convert it to rows before halving:

$$s_{\rm rows}=s_{\rm KiB}/w,\qquad B_j=\max(1,\lceil s_{\rm rows}/2^j\rceil).$$

Stop at one row. Also retain widths $\lceil\alpha s_{\rm rows}\rceil$ for $\alpha\in\{1,1.25,1.5,1.75,2\}$. Keep the first occurrence of each integer width, considering the coarse branch first. Shapes can have different numbers of widths. The realized size is $Bw$ KiB.

At $B=1$, ranking full tiles is stable Top-$R$: each tile is a row and the residual is zero. The endpoint is labeled `Top-R` and measured once. Whether finer blocks improve selection-to-read time is determined by the measurements.

In [4]:
curve = pd.read_csv(PROFILE).groupby("size_kib").latency_ms.median().sort_index()
assert np.isfinite(curve).all() and (curve > 0).all()
boundary, ceiling = float((curve / curve.index).idxmin()), float(curve.index.max())

In [5]:
def tile_grid(s):
    levels = max(0, int(np.ceil(np.log2(max(1, s)))))
    scales = [(f"{a:g}s", float(a)) for a in COARSE_SCALES]
    scales += [(f"s/2^{j}", 2.**-j) for j in range(1, levels + 1)]
    seen, grid = set(), []
    for label, scale in scales:
        width = max(1, int(np.ceil(scale * s)))
        if width not in seen:
            name = "Top-R" if width == 1 else f"Tiles ({label})"
            grid.append((name, scale, width))
            seen.add(width)
    return grid

## Selecting tiles

For width $B$, consider offsets $0$ and $\lfloor B/2\rfloor$. At each offset, select the $\lfloor R/B\rfloor$ full tiles with greatest importance, then the best disjoint interval of length $R\bmod B$. Discard infeasible offsets and keep the mask with the larger importance-to-modeled-cost ratio. If $R<B$, select the best length-$R$ interval. At $R=N$, the measurement bypasses selection.

The following C++ cells build one temporary source file in order. The first creates it and the others append to it; rerunning this section starts with the first source cell. Compilation occurs before timing.

In [6]:
source_dir = TemporaryDirectory(prefix="selection03-")
tile_source_path = Path(source_dir.name) / "tiles.cpp"

The input checks require contiguous CPU tensors: float32 importance and a float64 cost lookup indexed by run length. Masks use one byte per row.

In [7]:
%%writefile $tile_source_path
#include <torch/extension.h>
#include <algorithm>
#include <cmath>
#include <numeric>
#include <tuple>
#include <vector>

using Result = std::tuple<torch::Tensor, double, double>;

namespace {
void check(const torch::Tensor& v, int64_t R, const torch::Tensor& T) {
    TORCH_CHECK(v.device().is_cpu() && v.scalar_type() == torch::kFloat32
                && v.dim() == 1 && v.is_contiguous(), "v must be contiguous CPU float32");
    TORCH_CHECK(0 <= R && R <= v.numel(), "R must be between 0 and N");
    TORCH_CHECK(T.device().is_cpu() && T.scalar_type() == torch::kFloat64
                && T.dim() == 1 && T.is_contiguous() && T.numel() > v.numel(),
                "T must be contiguous CPU float64 with N+1 entries");
}

Writing /tmp/selection03-c6zya334/tiles.cpp


Full tiles are ranked by summed importance. Equal scores favor the earlier tile; the order of the selected disjoint tiles is immaterial.

In [8]:
%%writefile -a $tile_source_path
// Select the largest k scores, breaking ties by the original index.
// Their internal order is irrelevant because the chosen tiles are disjoint.
std::vector<int64_t> largest(const std::vector<double>& scores, int64_t k) {
    std::vector<int64_t> order(scores.size());
    std::iota(order.begin(), order.end(), 0);
    auto better = [&](int64_t a, int64_t b) {
        return scores[a] != scores[b] ? scores[a] > scores[b] : a < b;
    };
    if (k > 0 && k < static_cast<int64_t>(order.size()))
        std::nth_element(order.begin(), order.begin() + k, order.end(), better);
    order.resize(k);
    return order;
}

Appending to /tmp/selection03-c6zya334/tiles.cpp


Adjacent selected rows form one run. The mask score sums importance over rows and modeled latency over these merged runs.

In [9]:
%%writefile -a $tile_source_path
// Report the selected importance and the cost of maximal, merged runs.
Result summarize(const torch::Tensor& v, torch::Tensor mask, const torch::Tensor& T) {
    const auto* values = v.data_ptr<float>();
    const auto* selected = mask.data_ptr<uint8_t>();
    const auto* cost = T.data_ptr<double>();
    double importance = 0, latency = 0;
    int64_t length = 0;
    for (int64_t i = 0; i < v.numel(); ++i) {
        if (selected[i]) {
            importance += values[i];
            ++length;
        } else {
            latency += cost[length];
            length = 0;
        }
    }
    return {mask, importance, latency + cost[length]};
}
}  // namespace

Appending to /tmp/selection03-c6zya334/tiles.cpp


Prefix sums give each interval’s importance. The selector fills the residual budget only in unoccupied rows, then compares the feasible offsets.

In [10]:
%%writefile -a $tile_source_path
Result tiles(torch::Tensor v, int64_t R, torch::Tensor T, int64_t B) {
    check(v, R, T);
    TORCH_CHECK(B >= 1, "tile width B must be positive");
    const int64_t n = v.numel();
    const auto* values = v.data_ptr<float>();
    std::vector<double> prefix(n + 1, 0);
    for (int64_t i = 0; i < n; ++i) prefix[i + 1] = prefix[i] + values[i];

    if (R < B) {
        int64_t start = 0;
        for (int64_t i = 1; i + R <= n; ++i)
            if (prefix[i + R] - prefix[i] > prefix[start + R] - prefix[start]) start = i;
        auto mask = torch::zeros({n}, torch::kUInt8);
        std::fill_n(mask.data_ptr<uint8_t>() + start, R, uint8_t{1});
        return summarize(v, mask, T);
    }

    const int64_t k = R / B, residual = R % B;
    Result best;
    double best_ratio = -1;
    for (int64_t offset : {int64_t{0}, B / 2}) {
        std::vector<int64_t> starts;
        std::vector<double> scores;
        for (int64_t start = offset; start + B <= n; start += B) {
            starts.push_back(start);
            scores.push_back(B == 1 ? values[start] : prefix[start + B] - prefix[start]);
        }
        if (static_cast<int64_t>(starts.size()) < k) continue;
        auto mask = torch::zeros({n}, torch::kUInt8);
        auto* selected = mask.data_ptr<uint8_t>();
        for (auto j : largest(scores, k))
            std::fill_n(selected + starts[j], B, uint8_t{1});

        if (residual) {
            std::vector<int64_t> occupied(n + 1, 0);
            for (int64_t i = 0; i < n; ++i) occupied[i + 1] = occupied[i] + selected[i];
            int64_t start = -1;
            double score = -1;
            for (int64_t i = 0; i + residual <= n; ++i) {
                double candidate = prefix[i + residual] - prefix[i];
                if (occupied[i + residual] == occupied[i] && candidate > score) {
                    start = i;
                    score = candidate;
                }
            }
            if (start < 0) continue;
            std::fill_n(selected + start, residual, uint8_t{1});
        }
        auto result = summarize(v, mask, T);
        double ratio = std::get<1>(result) / std::get<2>(result);
        if (ratio > best_ratio) {
            best_ratio = ratio;
            best = result;
        }
    }
    TORCH_CHECK(best_ratio >= 0, "no exact-R tiling was feasible");
    return best;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("tiles", &tiles);
}

Appending to /tmp/selection03-c6zya334/tiles.cpp


## Native dependencies

Paper generates windows at several sizes, sorts importance/cost scores on the GPU, then accepts non-overlapping windows on the CPU while they fit $R$. It may underfill. The shared reader merges adjacent rows and returns their bytes on the CPU using direct I/O. These library calls are used unchanged for every tile scale; the tile selector above stays on the CPU.

In [11]:
import torch
from torch.utils.cpp_extension import load
os.environ.setdefault("MAX_JOBS", "1")
torch.set_num_threads(1)
sys.path.insert(0, str((UPSTREAM / "src").resolve()))
import vlmflash
from vlmflash._native import native
assert Path(vlmflash.__file__).resolve().is_relative_to(UPSTREAM.resolve())
assert torch.cuda.is_available(), "CUDA is required for Paper's GPU sort."
reader = native()
assert reader is not None, "Native reader required."
selectors = load("vlmflash_selection03_notebook", [str(tile_source_path)],
                 extra_cflags=["-O3", "-std=c++17"], verbose=True)
table = vlmflash.LatencyTable(curve.to_dict())

[1/2] c++ -MMD -MF tiles.o.d -DTORCH_EXTENSION_NAME=vlmflash_selection03_notebook -DTORCH_API_INCLUDE_EXTENSION_H -isystem /home/yabsed/miniconda3/lib/python3.13/site-packages/torch/include -isystem /home/yabsed/miniconda3/lib/python3.13/site-packages/torch/include/torch/csrc/api/include -isystem /home/yabsed/miniconda3/include/python3.13 -fPIC -std=c++17 -O3 -std=c++17 -c /tmp/selection03-c6zya334/tiles.cpp -o tiles.o 
[2/2] c++ tiles.o -shared -L/home/yabsed/miniconda3/lib/python3.13/site-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o vlmflash_selection03_notebook.so


## Paper window tuning

The Appendix H procedure is evaluated on each shape: random importance with seed 0, sparsity 0.1, 30 repetitions, and start/stride caps from 4 to 64 KiB in 4 KiB steps. Equivalent row-quantized configurations are tested once. Candidate widths advance by the start width in rows and stop below $\lfloor(z_{\max}+1)/w\rfloor$, where $z_{\max}$ is the largest profiled size; their stride is capped by the tuned stride.

Tuning includes the library wrapper's setup and selects the finest configuration within 1.6 ms, then within 2 ms if needed. If none meets 2 ms, it takes the fastest measured configuration. These tuning times differ in scope from the prepared calls used in the comparison.

In [12]:
def tuning_trials(n, d):
    w, seen = d * weight_bytes / 1024, set()
    values = torch.rand(n, generator=torch.Generator().manual_seed(0)) + 0.5
    for start in range(4, 68, 4):
        for jump in range(4, 68, 4):
            row_config = (max(1, int(start / w)), max(1, int(jump / w)))
            if row_config in seen:
                continue
            seen.add(row_config)
            params = vlmflash.ChunkParams(start_kb=start, jump_cap_kb=jump)
            select = lambda: vlmflash.select_chunks(values, n-int(n*0.1), w, table, params, impl="native")
            try:
                select()
            except ValueError:
                continue
            times = []
            for _ in range(30):
                begin = perf_counter_ns()
                select()
                times.append((perf_counter_ns()-begin)/1e6)
            yield dict(start_kb=start, jump_cap_kb=jump, overhead_ms=round(float(np.median(times)), 3))

The tuning result records the chosen parameters and the number of configurations inside the 2 ms budget.

In [13]:
def tune_windows():
    tuned = {}
    for shape in sorted({t["shape"] for t in traces}):
        trials = list(tuning_trials(*map(int, shape.split("x"))))
        assert trials, f"No evaluable Paper windows for {shape}."
        within = [t for t in trials if t["overhead_ms"] <= 2]
        pool = [t for t in within if t["overhead_ms"] <= 1.6] or within
        best = (min(pool, key=lambda t: (t["start_kb"] + t["jump_cap_kb"], t["start_kb"]))
                if pool else min(trials, key=lambda t: t["overhead_ms"]))
        tuned[shape] = dict(best=best, num_feasible=len(within), num_evaluated=len(trials))
        print(f"{shape}: {best}, {len(within)}/{len(trials)} within 2 ms", flush=True)
    return tuned

## Preparing one trace

Every selector shares normalized importance and the same cost lookup. Row width, candidate windows and cost tensors are prepared outside timing. The table of tile widths is also fixed before timing; only selection and reading are measured.

In [14]:
def paper_windows(n, w, tuned):
    start = max(1, int(tuned["start_kb"] / w))
    stride = max(1, int(tuned["jump_cap_kb"] / w))
    if ceiling < 2*w:
        return torch.tensor([1]), torch.tensor([table.read_ms(w)], dtype=torch.float64), 1
    sizes = list(range(start, min(n+1, int((ceiling+1)/w)), start))
    assert sizes, "Paper window range is empty."
    return (torch.tensor(sizes),
            torch.tensor([table.read_ms(w*r) for r in sizes], dtype=torch.float64), stride)

In [15]:
def prepare(trace):
    v = trace["values"].astype(np.float32)
    assert v.ndim == 1 and len(v) == trace["n"] and np.isfinite(v).all() and (v >= 0).all()
    values = torch.from_numpy(v / v.mean() if v.any() else v)
    row_bytes = weight_bytes * trace["d"]
    w = row_bytes / 1024
    costs = torch.tensor([0.] + [table.read_ms(w*r) for r in range(1, len(v)+1)], dtype=torch.float64)
    sizes, delays, stride = paper_windows(len(v), w, tuning[trace["shape"]]["best"])
    grid = tile_grid(boundary / w)
    widths = {name: B for name, a, B in grid}
    scales = {name: a for name, a, B in grid}
    methods = {PAPER: lambda R: reader.select_chunks(values, R, sizes, delays, stride, True)[0]}
    methods.update({name: lambda R, B=B: selectors.tiles(values, R, costs, B)[0]
                    for name, B in widths.items()})
    return values, row_bytes, widths, costs, methods, scales

Before timing, each mask is checked against its budget. Its selected intervals, importance, fill and modeled cost are recorded once and reused across repetitions.

In [16]:
def describe(mask, values, R, costs, method):
    selected, total = int(mask.sum()), float(values.double().sum())
    assert selected <= R if method == PAPER else selected == R
    spans = np.flatnonzero(np.diff(np.r_[False, mask.numpy().astype(bool), False])).reshape(-1, 2)
    return dict(R_selected=selected, fill_pct=100*selected/R,
        retained_pct=100*float(values[mask.bool()].double().sum())/total if total else np.nan,
        modeled_io_ms=float(costs[np.diff(spans).ravel()].sum()), runs=len(spans),
        intervals=json.dumps(spans.tolist()))

## Timing one selection and read

Let $t_0$ precede selection, $t_1$ follow it, and $t_2$ follow reader return:

$$t_{\rm selection}=t_1-t_0,\qquad t_{\rm wall}=t_2-t_0.$$

Native `io_ms` measures its read/thread interval; wall time also includes read preparation. Paper's GPU score transfers are included in selection. GPU weight transfer and model computation are excluded. Reads merge adjacent rows, use `THREADS` threads and split requests at 768 KiB. Validation follows the timed interval.

In [17]:
def timed_read(select, R, expected, blob, row_bytes):
    full = R == expected.numel()
    begin = perf_counter_ns()
    mask = expected if full else select(R)
    after_select = perf_counter_ns()
    raw, io_us, _, direct = reader.read_rows(blob, mask, row_bytes, "cpu", THREADS, 768*1024, True)
    end = perf_counter_ns()
    assert direct, "O_DIRECT unavailable; refusing page-cache timings."
    assert torch.equal(mask, expected) and raw.shape == (int(expected.sum()), row_bytes)
    return dict(selection_ms=(after_select-begin)/1e6, io_ms=io_us/1000, wall_ms=(end-begin)/1e6)

## Sweeping the budget and tile scale

For each trace and fraction, run `WARMUP` rounds followed by `REPEATS` recorded rounds, randomly interleaving Paper and all distinct tile widths for that shape. They share the trace, requested budget, blob, reader and timing boundary. The all-rows case reuses a prepared dense mask. Every repetition records its scale and realized tile width; Paper has no fixed tile width.

In [18]:
def measure_trace(trace, blob, rng):
    values, row_bytes, widths, costs, methods, scales = prepare(trace)
    N, dense = len(values), torch.ones(len(values), dtype=torch.uint8)
    for fraction in FRACTIONS:
        R, checked = max(1, int(round(float(fraction), 6)*N)), {}
        for name, select in methods.items():
            mask = dense if R == N else select(R)
            if widths.get(name) == 1:
                expected = torch.zeros(N, dtype=torch.uint8)
                expected[torch.argsort(values, descending=True, stable=True)[:R]] = 1
                assert torch.equal(mask, expected), "One-row tiles must equal stable Top-R."
            checked[name] = (mask.clone(), describe(mask, values, R, costs, name))
        for repeat in range(-WARMUP, REPEATS):
            for name in rng.permutation(list(methods)):
                mask, description = checked[name]
                timing = timed_read(methods[name], R, mask, blob, row_bytes)
                if repeat >= 0:
                    yield dict(trace_id=trace["trace_id"], shape=trace["shape"], fraction=float(fraction),
                        R_nominal=R, tile_scale=scales.get(name, np.nan),
                        tile_rows=widths.get(name, np.nan), tile_kib=widths.get(name, np.nan)*row_bytes/1024,
                        method=name, repeat=repeat, **timing, **description)

A random file is allocated on the target filesystem and flushed before measurement. The shared benchmark lock prevents cooperating experiments from timing this drive concurrently.

In [19]:
def write_blob(path, size):
    with open(path, "wb") as handle:
        os.posix_fallocate(handle.fileno(), 0, size)
        for begin in range(0, size, 2**20):
            handle.write(os.urandom(min(2**20, size-begin)))
        handle.flush()
        os.fsync(handle.fileno())

In [20]:
rng, records = np.random.default_rng(0), []
with (open(os.environ.get("VLMFLASH_LOCK", "/tmp/vlmflash-bench.lock"), "a+") as lock,
      TemporaryDirectory(dir=RESULTS.parent) as work):
    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    tuning = tune_windows()
    blob = str(Path(work) / "rows.dat")
    write_blob(blob, max(t["n"] * t["d"] * weight_bytes for t in traces))
    for index, trace in enumerate(traces, 1):
        records.extend(measure_trace(trace, blob, rng))
        if index % 16 == 0 or index == len(traces):
            print(f"{index}/{len(traces)} traces measured", flush=True)
pd.DataFrame(records).to_csv(RESULTS, index=False)

4864x896: {'start_kb': 12, 'jump_cap_kb': 8, 'overhead_ms': 1.53}, 245/256 within 2 ms
896x128: {'start_kb': 4, 'jump_cap_kb': 4, 'overhead_ms': 0.156}, 256/256 within 2 ms
896x4864: {'start_kb': 4, 'jump_cap_kb': 4, 'overhead_ms': 1.146}, 36/36 within 2 ms
896x896: {'start_kb': 4, 'jump_cap_kb': 4, 'overhead_ms': 1.369}, 256/256 within 2 ms
16/128 traces measured
32/128 traces measured
48/128 traces measured
64/128 traces measured
80/128 traces measured
96/128 traces measured
112/128 traces measured
128/128 traces measured


## Measurement record

Save the device, tuning and acquisition settings alongside the CSV. These describe the measurement session used by the report.

In [21]:
environment = dict(platform=platform.platform(), torch=torch.__version__, cuda_available=torch.cuda.is_available(),
    paper_sort_devices=["cuda"], paper_tuning=tuning,
    torch_threads=torch.get_num_threads(), reader_threads=THREADS, direct_io=True, destination="cpu",
    tile_boundary_kib=boundary, coarse_scales=COARSE_SCALES, dtype_bytes=weight_bytes,
    tile_grid="halving_to_one_row; unique widths per shape",
    tile_offsets="0, floor(B/2)", paper_ceiling_kib=ceiling, traces=len(traces),
    warmup=WARMUP, repeats=REPEATS, fractions=FRACTIONS.tolist(), max_read_kib=768,
    measurement="prepared CPU importance through selected bytes on CPU; no model compute",
    upstream_commit=subprocess.check_output(["git","-C",str(UPSTREAM),"rev-parse","HEAD"],text=True).strip(),
    mount=subprocess.check_output(["findmnt","-T",str(RESULTS.resolve()),"-n","-o","SOURCE,FSTYPE,TARGET"],text=True).strip())
ENVIRONMENT.write_text(json.dumps(environment, indent=2))

1617

The CSV contains every recorded repetition, the selected intervals and the realized tile widths. The JSON records the device, tuning and acquisition settings. Together they define the measured session; comparisons and conclusions are computed from these saved values in the analysis notebook.